In [ ]:
import joblib

df = joblib.load("dnd_df.pkl")
model = joblib.load("dnd_rf_model.pkl")
vectorizer = joblib.load("dnd_vectorizer.pkl")
metadata_features = joblib.load("metadata_features.pkl")

In [ ]:
print(type(model))
print(type(vectorizer))
print(metadata_features)

# Probability Function

In [ ]:
from scipy.sparse import csr_matrix, hstack

def score_message(
    message,
    sender_importance,
    is_group_chat,
    hour_of_day,
    average_messages_per_day
):

    X_text = vectorizer.transform([message])

    X_meta = csr_matrix([[
        sender_importance,
        is_group_chat,
        hour_of_day,
        average_messages_per_day
    ]])

    X_combined = hstack([X_text, X_meta])

    probability = model.predict_proba(X_combined)[0][1]

    return probability

In [ ]:
#test
score_message(
    message="Call me ASAP",
    sender_importance=3,
    is_group_chat=0,
    hour_of_day=2,
    average_messages_per_day=1
)

# Decision Engine

In [ ]:
def notification_action(prob):

    if prob >= 0.85:
        return "BYPASS_DND"

    elif prob >= 0.65:
        return "SILENT_SUMMARY"

    else:
        return "SUPPRESS"

In [ ]:
#test
prob = score_message(
    "Call me ASAP",
    3,
    0,
    2,
    1
)

action = notification_action(prob)

print(prob)
print(action)

# Train the engine on sample data

In [ ]:
import pandas as pd


# Take a sample for evaluation
decision_results = df.copy()

# Generate probabilities
decision_results['probability'] = decision_results.apply(
    lambda row: score_message(
        message=row['message'],
        sender_importance=row['sender_importance'],
        is_group_chat=row['is_group_chat'],
        hour_of_day=row['hour_of_day'],
        average_messages_per_day=row['average_messages_per_day']
    ),
    axis=1
)

# Generate actions
decision_results['action'] = decision_results['probability'].apply(
    notification_action
)

# Keep only useful columns
decision_results = decision_results[
    [
        'message',
        'label',
        'probability',
        'action'
    ]
]

decision_results.head(10)

In [ ]:
decision_results['action'].value_counts()

In [ ]:
decision_results[
    decision_results['action'] == 'BYPASS_DND'
].head(5)

In [ ]:
decision_results[
    decision_results['action'] == 'SILENT_SUMMARY'
].head(5)

In [ ]:
decision_results[
    decision_results['action'] == 'SUPPRESS'
].head(5)

In [ ]:
pd.crosstab(
    decision_results['label'],
    decision_results['action'],
    margins=True
)

Ten falses negative

In [ ]:
decision_results[
    (decision_results['label'] == 1)
    & (decision_results['action'] == 'SUPPRESS')
]

THis current policy is very conservative. All false negatives i.e. actually urgent messages should have bypassed. Need to re-evaluate thresholds.

# Threshold estimation

Probability Distribution Analysis

In [ ]:
import matplotlib.pyplot as plt

plt.hist(
    decision_results[
        decision_results["label"] == 0
    ]["probability"],
    bins=20,
    alpha=0.5,
    label="Non-Urgent"
)

plt.hist(
    decision_results[
        decision_results["label"] == 1
    ]["probability"],
    bins=20,
    alpha=0.5,
    label="Urgent"
)

plt.legend()
plt.xlabel("Predicted Probability")
plt.ylabel("Count")
plt.title("Probability Distribution by Class")
plt.show()

In [ ]:
decision_results.groupby("label")[
    "probability"
].describe()

Non-Urgent max = 0.6398, Urgent 25th percentile = 0.6638.

This becomes the urgent-non urgent class boundary separation: 0.6398 - 0.6638

Precision-Recall Curve

In [ ]:
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt

precision, recall, thresholds = (
    precision_recall_curve(
        decision_results["label"],
        decision_results["probability"]
    )
)

plt.figure(figsize=(8,5))
plt.plot(
    thresholds,
    precision[:-1],
    label="Precision"
)

plt.plot(
    thresholds,
    recall[:-1],
    label="Recall"
)

plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Precision-Recall vs Threshold")
plt.legend()
plt.grid(True)

plt.show()

Threshold Sweep

In [ ]:
import numpy as np
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

results = []

for threshold in np.arange(0.1, 1.0, 0.05):

    preds = (
        decision_results["probability"]
        >= threshold
    ).astype(int)

    results.append({
        "threshold": round(threshold, 2),
        "precision": precision_score(
            decision_results["label"],
            preds
        ),
        "recall": recall_score(
            decision_results["label"],
            preds
        ),
        "f1": f1_score(
            decision_results["label"],
            preds
        )
    })

threshold_df = pd.DataFrame(results)

threshold_df

In [ ]:
threshold_df.sort_values(
    "f1",
    ascending=False
)

We need two threshold boundaries:
1. Suppress vs. Urgent
2. Silent Summary vs. Bypass within the Urgent class

**Boundary 1**

From the threshold sweep, we can tell that 0.40-0.45 has the highest f1 score with extremely strong precision and recall values around 97 and 95 percent respectively.

Therefore, the model is highly confident that <0.40 probability messages will be non-urgent

Therefore, we classify the Suppress threshold to be <0.40

**Boundary 2**

Non-Urgent max - 0.6398

Therefore, a threshold of 0.65 seems reasonable because there's no instances of non-urgent messages beyond that threshold.

This can be backed by the Q1 value for urgent messages which lies around 0.66.

**Final Boundaries**

SUPPRESS: <0.40

SILENT_SUMMARY: 0.40 - 0.65

BYPASS: >0.65

Thresholds were selected using a combination of probability distribution analysis and precision-recall evaluation. A score of 0.40 was chosen as the lower boundary because it corresponded to the optimal precision-recall tradeoff (F1 = 0.964). A score of 0.65 was selected as the bypass threshold because it lay above the maximum probability assigned to any non-urgent message in the labeled dataset, making it a high-confidence urgency indicator. Messages falling between these boundaries were routed to a silent summary state, representing uncertain cases that may warrant later attention without interrupting the user.